In [1]:
import numpy as np
from pathlib import Path
import pickle
from tqdm import tqdm
from modules import background_model, ModelDataSet, SequenceRepresentation

SEED=42

2025-07-11 08:41:03.334676: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-11 08:41:03.484350: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-11 08:41:05.746154: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-11 08:41:06.220406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752223267.586829  551670 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752223267.90

In [2]:
datadir = Path("/home/ebelm/brain/genomegraph/data/20250408_STREME_benchmark_revisited/diluted_dataset/1.00/primary_sequences/")
fastas = [fa for fa in datadir.glob("wgEncode*")]

In [3]:
def train_Q(fasta: Path, order: int, num_models: int,
            lr:float=0.01, lr_factor:float=0.75, lr_patience:int=10, epochs:int=50):
    data = ModelDataSet.ModelDataSet([SequenceRepresentation.Genome([seq]) for seq in SequenceRepresentation.loadFasta_agnostic(fasta)],
                                     ModelDataSet.DataMode.DNA,
                                     tile_size=100, tiles_per_X=1, batch_size=1, prefetch=2)
    bgmod = background_model.TrainedQ(data, num_models=num_models, order=order, rand_seed=SEED)
    bgmod.train(lr=lr, lr_factor=lr_factor, lr_patience=lr_patience, epochs=epochs)

    return bgmod.getQ(), bgmod.getM(), data.alphabet

In [4]:
def print_model(Q, m, alphabet):
    assert len(Q.shape) >= 2, f"{Q.shape}"
    assert all(d == len(alphabet) for d in Q.shape[1:]), f"{Q.shape}"
    assert Q.shape[0] == len(m), f"{Q.shape}, {len(m)}"
    order = len(Q.shape[1:]) - 1
    if order > 2:
        raise RuntimeError("Order > 2 not supported")

    for i in range(len(m)):
        model = Q[i]
        weight = m[i]

        print(f"Background Model {i+1}/{len(m)}")
        print()
        if order == 0:
            for j in range(len(alphabet)):
                c = alphabet[j]
                print(f"     {c}: {model[j]:.2f}")

            print("------------")
            print(f"weight: {weight:.2f}")
            print("")

        else:
            def _get_lines_o1(Q1, w):
                lines = []
                lines.append(f"    {'    '.join(alphabet)}")
                lines.append(f"--+-{'----'.join(['-' for _ in alphabet])}---")
                for j in range(len(alphabet)):
                    c = alphabet[j]
                    lines.append(f"{c} | {' '.join(f'{q:.2f}' for q in Q1[j])}")

                lines.append(f"weight: {w:.2f}")
                lines.append('')
                return lines

            if order == 1:
                lines = _get_lines_o1(model, weight)
                for line in lines:
                    print(line)
            
            if order == 2:
                lines = None
                for k in range(len(alphabet)):
                    klines = _get_lines_o1(model[k], weight)
                    klines[0] = f"{alphabet[k]}  "+klines[0]
                    for l in range(1, len(klines)):
                        klines[l] = "   "+klines[l]

                    mlen = max([len(ln) for ln in klines])
                    for l in range(len(klines)):
                        klines[l] = klines[l] + " "*(mlen-len(klines[l]))

                    if lines is not None:
                        klines[-2] = " "*len(klines[-2])
                        for l in range(len(klines)-2):
                            lines[l] = lines[l] + "  |  " + klines[l]
                        lines[-2] = lines[-2] + "     " + klines[-2]
                        lines[-1] = lines[-1] + "     " + klines[-1]
                    else:
                        lines = klines.copy()

                for line in lines:
                    print(line)


In [5]:
def count_model(fasta: Path, order: int):
    model, freqs, alphabet = background_model.get_background_model(order, model_type="data", src=fasta)
    print_model(np.array([model]), [1], alphabet)

Train Background Models

In [9]:
fasta = fastas[0]
num_models = 3
Q_0, m_0, alphabet = train_Q(fasta, 0, num_models)
Q_1, m_1, alphabet = train_Q(fasta, 1, num_models)
Q_2, m_2, alphabet = train_Q(fasta, 2, num_models)

2025-07-10 23:04:56.053904: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-07-10 23:06:06.160781: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [10]:
print(f"Fasta: {fasta.name}")
print_model(Q_0, m_0, alphabet)

Fasta: wgEncodeAwgTfbsSydhK562Rfx5IggrabUniPk.narrowPeak.fasta
Background Model 1/3

     A: 0.25
     C: 0.25
     G: 0.30
     T: 0.21
------------
weight: 0.26

Background Model 2/3

     A: 0.16
     C: 0.34
     G: 0.32
     T: 0.18
------------
weight: 0.45

Background Model 3/3

     A: 0.30
     C: 0.20
     G: 0.19
     T: 0.32
------------
weight: 0.29



In [27]:
print("=== COUNTED NTs ===")
count_model(fastas[0], 0)

COUNTED NTs
Background Model 1/1

     A: 0.22
     C: 0.28
     G: 0.28
     T: 0.22
------------
weight: 1.00



In [11]:
print_model(Q_1, m_1, alphabet)

Background Model 1/3

    A    C    G    T
--+--------------------
A | 0.36 0.16 0.19 0.29
C | 0.31 0.21 0.14 0.34
G | 0.29 0.21 0.21 0.29
T | 0.25 0.17 0.21 0.37
weight: 0.22

Background Model 2/3

    A    C    G    T
--+--------------------
A | 0.25 0.22 0.34 0.18
C | 0.30 0.29 0.11 0.30
G | 0.24 0.26 0.30 0.20
T | 0.15 0.26 0.34 0.25
weight: 0.37

Background Model 3/3

    A    C    G    T
--+--------------------
A | 0.19 0.29 0.36 0.15
C | 0.17 0.33 0.31 0.19
G | 0.17 0.37 0.32 0.15
T | 0.14 0.33 0.33 0.20
weight: 0.41



In [28]:
print("=== COUNTED NTs ===")
count_model(fastas[0], 1)

COUNTED NTs
Background Model 1/1

    A    C    G    T
--+--------------------
A | 0.26 0.23 0.31 0.20
C | 0.25 0.30 0.21 0.25
G | 0.21 0.30 0.30 0.18
T | 0.17 0.26 0.30 0.27
weight: 1.00



In [12]:
print_model(Q_2, m_2, alphabet)

Background Model 1/3

A      A    C    G    T     |  C      A    C    G    T     |  G      A    C    G    T     |  T      A    C    G    T   
   --+--------------------  |     --+--------------------  |     --+--------------------  |     --+--------------------
   A | 0.20 0.28 0.34 0.18  |     A | 0.18 0.29 0.38 0.15  |     A | 0.19 0.30 0.36 0.15  |     A | 0.20 0.25 0.38 0.16
   C | 0.19 0.28 0.36 0.18  |     C | 0.18 0.31 0.31 0.20  |     C | 0.15 0.33 0.34 0.18  |     C | 0.22 0.39 0.19 0.20
   G | 0.16 0.34 0.33 0.16  |     G | 0.15 0.37 0.32 0.16  |     G | 0.20 0.36 0.31 0.14  |     G | 0.22 0.30 0.33 0.15
   T | 0.14 0.32 0.30 0.24  |     T | 0.15 0.33 0.35 0.17  |     T | 0.14 0.34 0.34 0.18  |     T | 0.14 0.33 0.32 0.21
   weight: 0.43                                                                                                        
                                                                                                                       
Background Model 2

In [29]:
print("=== COUNTED NTs ===")
count_model(fastas[0], 2)

=== COUNTED NTs ===
Background Model 1/1

A      A    C    G    T     |  C      A    C    G    T     |  G      A    C    G    T     |  T      A    C    G    T   
   --+--------------------  |     --+--------------------  |     --+--------------------  |     --+--------------------
   A | 0.30 0.22 0.26 0.22  |     A | 0.23 0.24 0.36 0.18  |     A | 0.25 0.25 0.33 0.17  |     A | 0.29 0.20 0.26 0.25
   C | 0.28 0.26 0.21 0.25  |     C | 0.25 0.29 0.22 0.24  |     C | 0.20 0.31 0.25 0.24  |     C | 0.28 0.32 0.13 0.27
   G | 0.24 0.28 0.30 0.18  |     G | 0.14 0.35 0.32 0.19  |     G | 0.23 0.32 0.29 0.16  |     G | 0.23 0.25 0.30 0.21
   T | 0.22 0.23 0.27 0.29  |     T | 0.14 0.28 0.36 0.22  |     T | 0.15 0.28 0.31 0.26  |     T | 0.18 0.25 0.26 0.31
   weight: 1.00                                                                                                        
                                                                                                                      

Train all fastas

In [10]:
pklfile = Path().resolve() / "20250710_learn_real_data_Q.pkl"
if pklfile.exists():
    with open(pklfile, 'rb') as fh:
        runs = pickle.load(fh)
else:
    runs = {}

num_models = 3
for fasta in tqdm(fastas):
    if fasta.name not in runs:
        runs[fasta.name] = {}
    
    for order in [0, 1, 2]:
    # for order in [0, 1]:
        if order not in runs[fasta.name]:
            _Q, _m, alphabet = train_Q(fasta, order, num_models)
            runs[fasta.name][order] = {
                'Q': _Q,
                'm': _m,
                'alphabet': alphabet
            }

            # save all intermediate steps in case the notebook dies again
            with open(pklfile, "wb") as fh:
                pickle.dump(runs, fh)
        else:
            print(f"Skipping existing {fasta.name} order {order}")

  0%|          | 0/40 [00:00<?, ?it/s]

Skipping existing wgEncodeAwgTfbsSydhK562Rfx5IggrabUniPk.narrowPeak.fasta order 0
Skipping existing wgEncodeAwgTfbsSydhK562Rfx5IggrabUniPk.narrowPeak.fasta order 1
Skipping existing wgEncodeAwgTfbsSydhK562Rfx5IggrabUniPk.narrowPeak.fasta order 2
Skipping existing wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak.fasta order 0
Skipping existing wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak.fasta order 1


2025-07-11 09:44:27.782189: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-07-11 09:45:27.640221: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-07-11 09:47:27.538352: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-07-11 09:51:28.634794: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-07-11 09:59:25.060728: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-07-11 10:15:20.259038: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
  5%|▌         | 2/40 [49:48<15:46:24, 1494.32s/it]

Skipping existing wgEncodeAwgTfbsSydhK562Znf143IggrabUniPk.narrowPeak.fasta order 0
Skipping existing wgEncodeAwgTfbsSydhK562Znf143IggrabUniPk.narrowPeak.fasta order 1


2025-07-11 10:46:20.574454: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
  8%|▊         | 3/40 [1:36:28<20:57:02, 2038.46s/it]

...

Skipping existing wgEncodeAwgTfbsSydhK562Nfe2UniPk.narrowPeak.fasta order 0
Skipping existing wgEncodeAwgTfbsSydhK562Nfe2UniPk.narrowPeak.fasta order 1


2025-07-11 11:56:24.330722: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
 18%|█▊        | 7/40 [2:16:41<6:57:12, 758.57s/it]

Skipping existing wgEncodeAwgTfbsSydhK562Elk112771IggrabUniPk.narrowPeak.fasta order 0
Skipping existing wgEncodeAwgTfbsSydhK562Elk112771IggrabUniPk.narrowPeak.fasta order 1


 20%|██        | 8/40 [2:21:24<5:24:49, 609.06s/it]

Skipping existing wgEncodeAwgTfbsHaibK562Gata2sc267Pcr1xUniPk.narrowPeak.fasta order 0
Skipping existing wgEncodeAwgTfbsHaibK562Gata2sc267Pcr1xUniPk.narrowPeak.fasta order 1


In [7]:
# pklfile = Path().resolve() / "20250710_learn_real_data_Q.pkl"
# if pklfile.exists():
#     with open(pklfile, 'rb') as fh:
#         runs = pickle.load(fh)

2025-07-11 08:54:32.266254: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
for name in runs:
    print(f"File {name}:")
    print()
    for order in [0, 1, 2]:
        if order not in runs[name]:
            continue
        
        print("Order {order}")
        print()
        print_model(runs[name][order]['Q'], runs[name][order]['m'], runs[name][order]['alphabet'])

    print()
    print()
    print()

File wgEncodeAwgTfbsSydhK562Rfx5IggrabUniPk.narrowPeak.fasta:

Order {order}

Background Model 1/3

     A: 0.25
     C: 0.25
     G: 0.30
     T: 0.21
------------
weight: 0.26

Background Model 2/3

     A: 0.16
     C: 0.34
     G: 0.32
     T: 0.18
------------
weight: 0.45

Background Model 3/3

     A: 0.30
     C: 0.20
     G: 0.19
     T: 0.32
------------
weight: 0.29

Order {order}

Background Model 1/3

    A    C    G    T
--+--------------------
A | 0.36 0.16 0.19 0.29
C | 0.31 0.21 0.14 0.34
G | 0.29 0.21 0.21 0.29
T | 0.25 0.17 0.21 0.37
weight: 0.22

Background Model 2/3

    A    C    G    T
--+--------------------
A | 0.25 0.22 0.34 0.18
C | 0.30 0.29 0.11 0.30
G | 0.24 0.26 0.30 0.20
T | 0.15 0.26 0.34 0.25
weight: 0.37

Background Model 3/3

    A    C    G    T
--+--------------------
A | 0.19 0.29 0.36 0.15
C | 0.17 0.33 0.31 0.19
G | 0.17 0.37 0.32 0.15
T | 0.14 0.33 0.33 0.20
weight: 0.41

Order {order}

Background Model 1/3

A      A    C    G    T     |  C    